# Zero-Error Solution-Method Analysis

This notebook runs the exact dynamic program and the greedy approximation on the saved 1D and 2D benchmarks. The exact minimum sequential sensor budget is denoted $N_{\min}^{\mathrm{seq}}$ throughout the analysis; solver-quality comparisons use $N_{\mathrm{greedy}}/N_{\min}^{\mathrm{seq}}$.

**Independent execution:** this notebook reads raw benchmark inputs and does not depend on another notebook. Keep `GENERATE_MISSING_INPUTS = False` to use the existing inputs. Choose `RUN_MODE = "test"` or `"full"` below, then run all cells. The default is `"test"`.

- `02_minimum_sensor_trends.ipynb` independently computes the $N_{\min}^{\mathrm{seq}}$ trends.
- `03_lower_bound_comparison.ipynb` independently computes and plots $N_{\mathrm{LB}}^{\mathrm{global}}$ against $N_{\min}^{\mathrm{seq}}$.
- Run these three analyses in any order, then run `final_figures.ipynb`, which reads their saved tables without running solvers.


In [ ]:
# Import notebook dependencies and resolve project paths for local modules.
from pathlib import Path
from collections import defaultdict
import csv
import math
import sys

from IPython.display import Image, Markdown, display

# Resolve paths whether the notebook starts from the repository root or this folder.
working_directory = Path.cwd().resolve()
PROJECT_DIRECTORY = (
    working_directory / "problem-1-zero-error"
    if (working_directory / "problem-1-zero-error").is_dir()
    else working_directory
)
SOURCE_DIRECTORY = PROJECT_DIRECTORY / "src"
SCRIPT_DIRECTORY = PROJECT_DIRECTORY / "scripts"
for directory in (SOURCE_DIRECTORY, SCRIPT_DIRECTORY):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))

# Reload local modules when the notebook is restarted after code changes.
local_modules = (
    "run_config", "optimal", "greedy", "phase_data", "field_models", "eval",
    "benchmark_analysis", "generate_inputs", "generate_benchmarks",
    "evaluate_benchmarks",
)
for module_name in local_modules:
    sys.modules.pop(module_name, None)

from evaluate_benchmarks import DATASETS, run_benchmark_evaluation
from benchmark_analysis import family_excess_table, guarantee_table, heatmap_table

print("Setup complete.")
print("Working directory:", working_directory)
print("Project directory:", PROJECT_DIRECTORY)
print("Source directory:", SOURCE_DIRECTORY)
print("Script directory:", SCRIPT_DIRECTORY)
print("Datasets queued:", ", ".join(DATASETS))


## Run configuration

- **`"test"` (default):** 96 cases across both dimensions, all three field families and all $K=3,\ldots,10$, using $M=3,6$ and replicate 1.
- **`"full"`:** all 11,520 saved benchmark cases. This may take several minutes.

Both modes use the same solvers. Test outputs are stored separately; final figures use full outputs only. Change the setting below and run all cells.

`REPEATS` affects runtime averaging only. `BOOTSTRAP_ITERATIONS` controls confidence intervals. Keep `GENERATE_MISSING_INPUTS = False` to use the saved inputs.


In [ ]:
RUN_MODE = "test"  # Change to "full" for all 11,520 benchmark cases.

if RUN_MODE not in ("test", "full"):
    raise ValueError('RUN_MODE must be "test" or "full"')
print("Run mode:", RUN_MODE)
print("Expected standard benchmark cases:", 96 if RUN_MODE == "test" else 11_520)

REPEATS = 1
BOOTSTRAP_ITERATIONS = 200 if RUN_MODE == "test" else 2_000
GENERATE_MISSING_INPUTS = False
print("Bootstrap iterations:", BOOTSTRAP_ITERATIONS)


## Run evaluation

Evaluate both dimensions using the selected mode and create the analysis tables and figures. Test results are previews based on 96 cases, not full benchmark findings.

In [ ]:
# Run both benchmark dimensions, then build the statistical analysis and figures.
print("Starting zero-error benchmark evaluation.")
print("The evaluator will print one progress block per dataset.")
analysis = run_benchmark_evaluation(
    run_mode=RUN_MODE,
    repeats=REPEATS,
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
    generate_missing=GENERATE_MISSING_INPUTS,
)
print("Evaluation complete.")
print("Analyzed rows:", len(analysis["rows"]))
print("Analysis output directory:", analysis["output_directory"])

# Stop if the analysis does not contain exactly the expected analysis figures.
ANALYSIS_FIGURES = ("families", "guarantee", "heatmaps")
if set(analysis["figures"]) != set(ANALYSIS_FIGURES):
    raise RuntimeError("The expected analysis figures were not created.")
print("Verified analysis figures:", ", ".join(ANALYSIS_FIGURES))


## Main Numerical Summary

The first row combines both dimensions. The following rows report 1D and 2D separately. The primary quality metric is the greedy-to-optimal sensor ratio, $N_{\mathrm{greedy}}/N_{\min}^{\mathrm{seq}}$.


In [ ]:
# Display the main solver-quality summary table for the full benchmark and each dimension.
def summary_table(rows):
    header = (
        "| Group | Cases | Mean ratio | 95% CI | Exact-match rate | "
        "95th percentile | Maximum |\n"
        "|---|---:|---:|---:|---:|---:|---:|"
    )
    body = []
    for row in rows:
        body.append(
            f"| {row['group_value']} | {row['case_count']} | "
            f"{row['mean_greedy_ratio']:.3f} | "
            f"[{row['mean_greedy_ratio_ci95_low']:.3f}, "
            f"{row['mean_greedy_ratio_ci95_high']:.3f}] | "
            f"{100 * row['exact_match_rate']:.1f}% | "
            f"{row['p95_greedy_ratio']:.3f} | "
            f"{row['max_greedy_ratio']:.3f} |"
        )
    return header + "\n" + "\n".join(body)

summary_rows = [analysis["overall_summary"], *analysis["dimension_summaries"]]
print(f"Displaying {len(summary_rows)} summary rows.")
display(Markdown(summary_table(summary_rows)))


## Figure Data Tables

These tables contain the numerical values used in the solution-method comparison figures. They are computed from the raw per-case results in analysis["rows"], not from image files.


In [ ]:
# Build and display the numeric tables behind the benchmark comparison figures.
def markdown_table(rows, columns, formats):
    header = "| " + " | ".join(label for label, _ in columns) + " |"
    separator = "|" + "|".join("---:" if formats.get(key) else "---" for _, key in columns) + "|"
    body = []
    for row in rows:
        values = []
        for _, key in columns:
            formatter = formats.get(key)
            value = row[key]
            values.append(formatter(value) if formatter else str(value))
        body.append("| " + " | ".join(values) + " |")
    return "\n".join([header, separator, *body])

heatmap_rows = heatmap_table(analysis["rows"])
print(f"Displaying quality heatmap table with {len(heatmap_rows)} rows.")
display(Markdown("### Quality Heatmap Values"))
display(Markdown(markdown_table(
    heatmap_rows,
    (
        ("Dimension", "dimension"),
        ("K", "K"),
        ("M", "M"),
        ("Mean additional sensors used by Greedy (%)", "mean_additional_sensors_percent"),
    ),
    {"K": lambda value: f"{int(value)}", "M": lambda value: f"{int(value)}", "mean_additional_sensors_percent": lambda value: f"{float(value):.2f}"},
)))

family_rows = family_excess_table(
    analysis["rows"],
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
)
print(f"Displaying field-family figure table with {len(family_rows)} rows.")
display(Markdown("### Field-Family Bar-Chart Values"))
display(Markdown(markdown_table(
    family_rows,
    (
        ("Dimension", "dimension"),
        ("Field family", "field_family"),
        ("K", "K"),
        ("Mean additional sensors used by Greedy (%)", "mean_additional_sensors_percent"),
        ("95% CI low", "ci95_low"),
        ("95% CI high", "ci95_high"),
    ),
    {
        "K": lambda value: f"{int(value)}",
        "mean_additional_sensors_percent": lambda value: f"{float(value):.2f}",
        "ci95_low": lambda value: f"{float(value):.2f}",
        "ci95_high": lambda value: f"{float(value):.2f}",
    },
)))

guarantee_rows = guarantee_table(analysis["rows"])
print(f"Displaying approximation-guarantee table with {len(guarantee_rows)} rows.")
display(Markdown("### Approximation-Guarantee Figure Values"))
display(Markdown(markdown_table(
    guarantee_rows,
    (
        ("Dimension", "dimension"),
        ("Field family", "field_family"),
        ("K", "K"),
        ("Maximum worst-case allowance used (%)", "maximum_worst_case_allowance_used_percent"),
    ),
    {"K": lambda value: f"{int(value)}", "maximum_worst_case_allowance_used_percent": lambda value: f"{float(value):.2f}"},
)))


## Solution-Method Comparison Figures

1. **Sensor excess by field family:** compares Gaussian, sinusoidal, and quadratic fields across K.
2. **Approximation guarantee comparison:** shows how much of the theoretical sensor-excess allowance was used.
3. **Quality heatmaps:** show mean Greedy sensor excess across K and M.

The figures use Times New Roman, 12 pt, and a color-blind-safe blue, green, and orange palette.


In [ ]:
# Display the solution-method comparison figures in the requested order.
PUBLICATION_FIGURES = ("families", "guarantee", "heatmaps")
plot_titles = {
    "families": "Sensor excess by field family",
    "guarantee": "Approximation guarantee comparison",
    "heatmaps": "Quality heatmaps",
}
plot_files = {
    "families": "benchmark_family_excess.png",
    "guarantee": "benchmark_guarantee_comparison.png",
    "heatmaps": "benchmark_quality_heatmaps.png",
}
figure_directory = Path(analysis["output_directory"]) / "figures"
print("Displaying solution-method figures from:", figure_directory)
for figure_key in PUBLICATION_FIGURES:
    figure_path = figure_directory / plot_files[figure_key]
    print(f"Displaying {figure_key}: {figure_path}")
    display(Markdown(f"### {plot_titles[figure_key]}"))
    display(Image(filename=str(figure_path)))


## Saved files

Outputs go to `results/methods/test/` or `results/methods/full/`, according to `RUN_MODE`. Combined tables and figures are in `tables/` and `figures/`; detailed per-dataset outputs are in `1d/tables/` and `2d/tables/`.

Notebooks 02 and 03 run independently in either mode. After running all three analyses in **full** mode, run `final_figures.ipynb`.